# Accidentalidad en Barranquilla: ¿el accidente deja víctimas?
**Fundamentos de Ciencia de Datos – Universidad de San Buenaventura**

**Autores:** Miguel Angel Tunjuelo Cortes, ladimir Rios


1. Definición del problema
2. Exploración de datos
3. Limpieza
4. Preparación
5. Modelado (3 modelos)
6. Evaluación
7. Selección del modelo
8. Conclusiones

**Dataset:** *Accidentalidad en Barranquilla* (Datos Abiertos Colombia), 28.865 accidentes de tránsito registrados entre 2018 y 2026.

## 1. Definición del problema

**¿Qué se quiere predecir?**
Queremos predecir si un accidente de tránsito en Barranquilla **deja víctimas (heridos o muertos)** o si es **solo de daños materiales**, a partir de información que se conoce en el momento del reporte: la clase de accidente, la hora, el día de la semana, el mes, el año y el tipo de vía donde ocurrió.

Es un problema de **clasificación binaria** (aprendizaje supervisado):
- `1` = accidente **con víctimas** (con heridos o con muertos)
- `0` = accidente **solo daños**

**¿Por qué es importante para Colombia?**
- Los siniestros viales son una de las principales causas de muerte violenta en Colombia, y la Agencia Nacional de Seguridad Vial (ANSV) los reconoce como un problema de salud pública.
- Si se puede estimar desde el primer reporte que un accidente probablemente tiene víctimas, las centrales de emergencia pueden **priorizar el envío de ambulancias** y de agentes de tránsito.
- Entender **en qué horarios, días y tipos de vía** los accidentes son más graves ayuda a la Alcaldía y a la Secretaría de Tránsito a diseñar **campañas y controles** donde más se necesitan.

### Librerías

In [2]:
# Librerias para manejo de datos
import pandas as pd
import numpy as np

# Librerias para gráficas
import matplotlib.pyplot as plt
import seaborn as sns

# Librerias de machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

import warnings
warnings.filterwarnings("ignore")

# Estilo de las gráficas
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 2. Exploración de datos
### 2.1 Carga del dataset

In [3]:
# Cargamos el archivo CSV con los accidentes
df = pd.read_csv("accidentalidad_barranquilla.csv")

# Tamaño del dataset: (filas, columnas)
print("Filas y columnas:", df.shape)

# Primeras 5 filas
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'accidentalidad_barranquilla.csv'

In [ ]:
# Renombramos las columnas para que sean más cortas y fáciles de usar
df.columns = ["fecha", "hora", "gravedad", "clase", "sitio",
              "heridos", "muertos", "cantidad", "ano", "mes", "dia"]

# Tipos de datos de cada columna
df.info()

### 2.1 Estadísticas

In [ ]:
# Estadísticas de las columnas numéricas (conteo, media, desviación, mínimo, cuartiles, máximo)
df.describe()

In [ ]:
# Estadísticas de las columnas categóricas (valores únicos y el más frecuente)
df.describe(include="object")

### 2.3 Detección de valores nulos

In [ ]:
# Cantidad y porcentaje de nulos por columna
nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "porcentaje (%)": (df.isna().mean() * 100).round(2)
})
nulos

In [ ]:
# ¿Los nulos de 'heridos' coinciden con algún tipo de gravedad?
pd.crosstab(df["gravedad"], df["heridos"].isna(), margins=True)

**Observación:** los nulos de `heridos` y `muertos` **no son datos perdidos al azar**: aparecen cuando el accidente es *solo daños* (no hubo heridos) o cuando hubo muertos pero no heridos. Es decir, un nulo significa **"cero personas"**. Esto se usa en la limpieza.

### 2.4 Distribuciones y gráficas

In [ ]:
# Distribución de la gravedad de los accidentes
fig, ax = plt.subplots(figsize=(8, 4))
orden = df["gravedad"].value_counts().index
sns.countplot(data=df, y="gravedad", order=orden, palette="Blues_r", ax=ax)
for c in ax.containers:
    ax.bar_label(c, padding=3)
ax.set_title("Accidentes según su gravedad")
ax.set_xlabel("Cantidad de accidentes"); ax.set_ylabel("")
plt.show()

In [ ]:
# Distribución por clase de accidente
fig, ax = plt.subplots(figsize=(8, 4))
orden = df["clase"].value_counts().index
sns.countplot(data=df, y="clase", order=orden, palette="Oranges_r", ax=ax)
for c in ax.containers:
    ax.bar_label(c, padding=3)
ax.set_title("Accidentes según la clase")
ax.set_xlabel("Cantidad de accidentes"); ax.set_ylabel("")
plt.show()

# Tabla cruzada: clase vs gravedad
pd.crosstab(df["clase"], df["gravedad"])

In [ ]:
# Accidentes por año
por_anio = df.groupby("anio").size()
fig, ax = plt.subplots()
por_anio.plot(kind="bar", color="steelblue", ax=ax)
ax.set_title("Accidentes por año en Barranquilla")
ax.set_xlabel("Año"); ax.set_ylabel("Cantidad")
plt.show()
print(por_anio)

In [ ]:
# Convertimos la hora (formato 01:30:00:am) a hora del día (0 a 23)
df["hora_dt"] = pd.to_datetime(df["hora"].str.strip(), format="%I:%M:%S:%p", errors="coerce")
df["hora_dia"] = df["hora_dt"].dt.hour

# Histograma de accidentes por hora del día
fig, ax = plt.subplots()
sns.histplot(df["hora_dia"], bins=24, discrete=True, color="teal", ax=ax)
ax.set_title("Distribución de accidentes por hora del día")
ax.set_xlabel("Hora (0-23)"); ax.set_ylabel("Cantidad")
ax.set_xticks(range(0, 24))
plt.show()

In [ ]:
# Porcentaje de accidentes CON víctimas según la hora
df["con_victimas"] = (df["gravedad"] != "Solo daños").astype(int)
pct_hora = df.groupby("hora_dia")["con_victimas"].mean() * 100

fig, ax = plt.subplots()
pct_hora.plot(marker="o", color="crimson", ax=ax)
ax.set_title("% de accidentes con víctimas según la hora")
ax.set_xlabel("Hora (0-23)"); ax.set_ylabel("% con víctimas")
ax.set_xticks(range(0, 24))
plt.show()

In [ ]:
# Accidentes y % con víctimas por día de la semana
orden_dias = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
resumen_dia = df.groupby("dia").agg(accidentes=("con_victimas", "size"),
                                    pct_victimas=("con_victimas", "mean")).loc[orden_dias]
resumen_dia["pct_victimas"] = (resumen_dia["pct_victimas"] * 100).round(1)

fig, ax = plt.subplots()
sns.barplot(x=resumen_dia.index, y=resumen_dia["pct_victimas"], color="salmon", ax=ax)
ax.set_title("% de accidentes con víctimas por día de la semana")
ax.set_xlabel("Día"); ax.set_ylabel("% con víctimas")
plt.show()
resumen_dia

In [ ]:
# % de accidentes con víctimas por año
pct_anio = (df.groupby("anio")["con_victimas"].mean() * 100).round(1)
fig, ax = plt.subplots()
pct_anio.plot(kind="bar", color="purple", ax=ax)
ax.set_title("% de accidentes con víctimas por año")
ax.set_xlabel("Año"); ax.set_ylabel("% con víctimas")
plt.show()
pct_anio

**Hallazgo importante:** entre 2018 y 2022 cerca del 30–49% de los accidentes registrados tienen víctimas, pero **desde 2023 casi el 100%** de los registros son con víctimas. No es que los accidentes se hayan vuelto más graves: a partir de 2023 los accidentes de **solo daños prácticamente dejaron de registrarse** en esta fuente (cambio en la forma de reportar). Esto se tiene en cuenta en la limpieza.

### 2.5 Detección de outliers (box plot)
Usamos diagramas de caja (*box plot*) y el método del **rango intercuartílico (IQR)** para detectar valores atípicos en la cantidad de heridos.

In [ ]:
# Box plot de la cantidad de heridos (solo accidentes que tuvieron heridos)
con_heridos = df[df["heridos"].notna()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=con_heridos["heridos"], color="lightblue", ax=axes[0])
axes[0].set_title("Box plot: heridos por accidente")
sns.boxplot(data=df, x="hora_dia", y="gravedad", palette="Set2", ax=axes[1])
axes[1].set_title("Box plot: hora del accidente según gravedad")
axes[1].set_xlabel("Hora (0-23)"); axes[1].set_ylabel("")
plt.tight_layout()
plt.show()

# Método IQR
Q1 = con_heridos["heridos"].quantile(0.25)
Q3 = con_heridos["heridos"].quantile(0.75)
IQR = Q3 - Q1
lim_sup = Q3 + 1.5 * IQR
outliers = con_heridos[con_heridos["heridos"] > lim_sup]
print(f"Q1 = {Q1}, Q3 = {Q3}, IQR = {IQR}, límite superior = {lim_sup}")
print(f"Accidentes con heridos por encima del límite (outliers): {len(outliers)}")
print("Máximo de heridos en un solo accidente:", con_heridos["heridos"].max())
outliers.sort_values("heridos", ascending=False)[["fecha", "clase", "sitio", "heridos"]].head(5)

**Observación:** hay accidentes con muchos heridos (por ejemplo, choques de buses), que son outliers estadísticos pero **son eventos reales**, no errores de digitación. Por eso **no se eliminan**. Además, la columna `heridos` no se usa como variable de entrada del modelo (ver sección 4), así que estos outliers no afectan el entrenamiento.

### 2.6 Regresión lineal: tendencia de accidentes en el tiempo
Antes de clasificar, usamos una **regresión lineal** simple para ver la tendencia de los accidentes mensuales a lo largo de los años.

In [ ]:
# Contamos accidentes por mes (serie mensual)
df["fecha_dt"] = pd.to_datetime(df["fecha"])
mensual = df.set_index("fecha_dt").resample("MS").size().reset_index(name="accidentes")
mensual["t"] = np.arange(len(mensual))   # índice de tiempo: 0, 1, 2, ...

# Ajustamos la regresión lineal: accidentes = a + b * t
reg = LinearRegression()
reg.fit(mensual[["t"]], mensual["accidentes"])
mensual["tendencia"] = reg.predict(mensual[["t"]])

print(f"Intercepto (a): {reg.intercept_:.2f}")
print(f"Pendiente (b): {reg.coef_[0]:.2f} accidentes por mes")
print(f"R²: {reg.score(mensual[['t']], mensual['accidentes']):.3f}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(mensual["fecha_dt"], mensual["accidentes"], label="Accidentes reportados", color="gray")
ax.plot(mensual["fecha_dt"], mensual["tendencia"], label="Tendencia (regresión lineal)", color="red", linewidth=2)
ax.set_title("Accidentes mensuales en Barranquilla y tendencia lineal")
ax.set_ylabel("Accidentes por mes"); ax.legend()
plt.show()

**Observación:** la pendiente es negativa: los accidentes **reportados** vienen bajando desde 2018. Se nota la caída de 2020 (pandemia) y otra caída fuerte desde 2023. Esta última coincide con el hallazgo de la sección 2.4 (desde 2023 casi no se registran accidentes de solo daños), así que la tendencia refleja en parte **cambios en el registro** y no solo una mejora real de la seguridad vial.

## 3. Limpieza
Decisiones tomadas (justificadas):

| Columna | Problema | Decisión | Justificación |
|---|---|---|---|
| `heridos` | 54% nulos | Imputar con **0** | El nulo significa que no hubo heridos (coincide con *Solo daños*). No se usa media ni mediana porque inventaría heridos que no existieron. |
| `muertos` | 98,8% nulos | Imputar con **0** | Mismo motivo: nulo = sin muertos. |
| `sitio` | Escrito de muchas formas (CL, CALLE, CLLE, CRA, KR...) | Normalizar a un **tipo de vía** | Se agrupan las variantes en Calle, Carrera, Avenida, Vía, Diagonal, Transversal u Otro. |
| `hora` | Texto `01:30:00:am` | Convertir a número 0–23 | Los modelos necesitan valores numéricos. |
| Duplicados | Filas repetidas | **Eliminar** | Un mismo accidente contado dos veces sesga el modelo. |
| Años 2023–2026 | Casi no hay registros de *solo daños* (≈ 99,6% con víctimas) | Usar solo **2018–2022** para el modelo | Si se incluyen, el modelo aprende que "año reciente = víctimas", que es un cambio en el registro y no una causa real. Sería un sesgo. |
| `clase` = *Incendio* | Solo 14 registros | Agrupar en **Otro** | Muy pocos casos para que el modelo aprenda algo de esa categoría (equivale a imputar con la **moda** del grupo "Otro"). |

In [ ]:
# 1) Nulos de heridos y muertos -> 0 (el nulo significa "ninguno")
df["heridos"] = df["heridos"].fillna(0).astype(int)
df["muertos"] = df["muertos"].fillna(0).astype(int)

# 2) Eliminar filas duplicadas
antes = len(df)
df = df.drop_duplicates(subset=["fecha", "hora", "gravedad", "clase", "sitio", "heridos", "muertos"])
print(f"Duplicados eliminados: {antes - len(df)}")

# 3) Agrupar clases con muy pocos registros
df["clase"] = df["clase"].replace({"Incendio": "Otro"})

# 4) Solo años 2018-2022 (en 2023+ casi no se registran accidentes de solo daños)
df = df[df["anio"] <= 2022].copy()
print("Filas 2018-2022:", len(df))

# 5) Filas sin hora válida (si las hubiera) -> se eliminan
print("Horas no válidas:", df["hora_dia"].isna().sum())
df = df.dropna(subset=["hora_dia"])

print("Filas después de la limpieza:", len(df))
print("Nulos restantes en heridos/muertos:", df[["heridos", "muertos"]].isna().sum().sum())

In [ ]:
# 6) Normalizar la dirección a un tipo de vía
def tipo_via(sitio):
    # Toma la primera palabra de la dirección y la agrupa en un tipo de vía
    primera = str(sitio).upper().replace(".", " ").split()
    primera = primera[0] if primera else ""
    if primera in ["CL", "CALLE", "CLLE", "CLL", "CALE", "CLL.", "CALL"]:
        return "Calle"
    if primera in ["CR", "CARRERA", "CRA", "KR", "CARERA", "CARERRA", "KRA", "CRR"]:
        return "Carrera"
    if primera in ["AV", "AVENIDA", "AVE", "CIRCUNVALAR", "CORDIALIDAD", "TRONCAL"]:
        return "Avenida"
    if primera in ["VIA", "KM", "KILOMETRO"]:
        return "Vía/Carretera"
    if primera in ["DG", "DIAGONAL", "DIAG"]:
        return "Diagonal"
    if primera in ["TV", "TRANSVERSAL", "TRANSV"]:
        return "Transversal"
    return "Otro"

df["tipo_via"] = df["sitio"].apply(tipo_via)
df["tipo_via"].value_counts()

## 4. Preparación
- **Variable objetivo (y):** `con_victimas` (1 = con heridos o muertos, 0 = solo daños).
- **Variables de entrada (X):** clase de accidente, tipo de vía, hora, franja horaria, día de la semana, fin de semana y mes.
- **Selección de variables:** se **excluyen** `heridos`, `muertos` y `gravedad` porque *son* la respuesta (si se usan, el modelo "hace trampa" — fuga de información o *data leakage*). También se excluyen `sitio` (texto libre, ya resumido en `tipo_via`) `cantidad` (casi siempre vale 1, no aporta información) y `anio` (no sirve para predecir accidentes futuros y arrastra el cambio en el registro).
- **Codificación:** las variables categóricas se transforman con **One-Hot Encoding**.
- **Escalado:** las variables numéricas se escalan con **StandardScaler** (media 0, desviación 1), necesario para la regresión logística.

In [ ]:
# Nuevas variables (feature engineering)
def franja(h):
    if 0 <= h < 6:   return "Madrugada"
    if 6 <= h < 12:  return "Mañana"
    if 12 <= h < 18: return "Tarde"
    return "Noche"

df["franja"] = df["hora_dia"].apply(franja)
df["fin_de_semana"] = df["dia"].isin(["Sat", "Sun"]).astype(int)
meses = {"January": 1, "February": 2, "March": 3, "April": 4, "May": 5, "June": 6, "July": 7,
         "August": 8, "September": 9, "October": 10, "November": 11, "December": 12}
df["mes_num"] = df["mes"].map(meses)

# Variable objetivo
df["con_victimas"] = (df["gravedad"] != "Solo daños").astype(int)

# Selección de variables
cat_cols = ["clase", "tipo_via", "franja", "dia"]
num_cols = ["hora_dia", "mes_num", "fin_de_semana"]
X = df[cat_cols + num_cols]
y = df["con_victimas"]

print("Variables de entrada:", cat_cols + num_cols)
print("\nBalance de la variable objetivo (%):")
print((y.value_counts(normalize=True) * 100).round(1))

Las clases están **algo desbalanceadas** (≈ 65% solo daños / 35% con víctimas). Para que los modelos no ignoren la clase minoritaria usamos `class_weight='balanced'`, que le da más peso a los accidentes con víctimas durante el entrenamiento.

In [ ]:
# División entrenamiento (80%) / prueba (20%), manteniendo la proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Entrenamiento:", X_train.shape, " Prueba:", X_test.shape)

# Preprocesador: One-Hot para categóricas y escalado para numéricas
preprocesador = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])

## 5. Modelado
Comparamos **3 modelos** de clasificación vistos en clase:
1. **Regresión logística**: modelo lineal que estima la probabilidad de que el accidente tenga víctimas.
2. **Árbol de decisión**: genera reglas tipo *si... entonces...* fáciles de interpretar.
3. **Random Forest**: conjunto de muchos árboles de decisión que votan; suele ser más preciso.

In [ ]:
modelos = {
    "Regresión logística": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Árbol de decisión":   DecisionTreeClassifier(max_depth=6, min_samples_leaf=50,
                                                  class_weight="balanced", random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=200, max_depth=12,
                                                  min_samples_leaf=10, class_weight="balanced",
                                                  random_state=42, n_jobs=-1),
}

pipelines = {}
for nombre, modelo in modelos.items():
    # Cada pipeline = preprocesamiento + modelo
    pipe = Pipeline([("prep", preprocesador), ("modelo", modelo)])
    pipe.fit(X_train, y_train)
    pipelines[nombre] = pipe
    print(f"✔ {nombre} entrenado")

## 6. Evaluación
Como es un problema de **clasificación**, usamos **Accuracy, Precision, Recall y F1**.
- **Recall** es la métrica más importante aquí: mide cuántos de los accidentes que *realmente* tuvieron víctimas detecta el modelo. Un falso negativo (decir "solo daños" cuando hay heridos) es el error más costoso, porque podría retrasar una ambulancia.

In [ ]:
resultados = []
for nombre, pipe in pipelines.items():
    pred = pipe.predict(X_test)
    resultados.append({
        "Modelo": nombre,
        "Accuracy":  accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall":    recall_score(y_test, pred),
        "F1":        f1_score(y_test, pred),
    })

tabla = pd.DataFrame(resultados).set_index("Modelo").round(3)
tabla

In [ ]:
# Gráfica comparativa de métricas
tabla.plot(kind="bar", figsize=(10, 5), colormap="viridis")
plt.title("Comparación de los 3 modelos")
plt.ylabel("Valor de la métrica"); plt.ylim(0, 1)
plt.xticks(rotation=0); plt.legend(loc="lower right")
plt.show()

In [ ]:
# Matrices de confusión de los 3 modelos
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (nombre, pipe) in zip(axes, pipelines.items()):
    cm = confusion_matrix(y_test, pipe.predict(X_test))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Solo daños", "Con víctimas"],
                yticklabels=["Solo daños", "Con víctimas"])
    ax.set_title(nombre); ax.set_xlabel("Predicción"); ax.set_ylabel("Real")
plt.tight_layout()
plt.show()

In [ ]:
# Visualización del árbol de decisión (primeros 3 niveles)
arbol = pipelines["Árbol de decisión"]
nombres_vars = arbol.named_steps["prep"].get_feature_names_out()
plt.figure(figsize=(20, 8))
plot_tree(arbol.named_steps["modelo"], feature_names=nombres_vars, max_depth=3,
          class_names=["Solo daños", "Con víctimas"], filled=True, fontsize=8)
plt.title("Árbol de decisión (primeros 3 niveles)")
plt.show()

## 7. Selección del modelo

In [ ]:
mejor = tabla["F1"].idxmax()
print("Modelo con mejor F1:", mejor)
print()
print(classification_report(y_test, pipelines[mejor].predict(X_test),
                            target_names=["Solo daños", "Con víctimas"]))

In [ ]:
# Importancia de las variables según el Random Forest
rf = pipelines["Random Forest"]
importancias = pd.Series(rf.named_steps["modelo"].feature_importances_,
                         index=rf.named_steps["prep"].get_feature_names_out())
importancias.index = importancias.index.str.replace("cat__", "").str.replace("num__", "")
top = importancias.sort_values(ascending=True).tail(12)

top.plot(kind="barh", color="darkorange", figsize=(9, 5))
plt.title("Variables más importantes (Random Forest)")
plt.xlabel("Importancia")
plt.show()

**Justificación:** se elige el **Árbol de decisión** para producción en la app de Streamlit porque:
- tiene el **mejor F1** de los tres modelos (≈ 0,51), es decir, el mejor equilibrio entre Precision y Recall;
- tiene el **mejor Recall** (≈ 0,53): detecta más accidentes que de verdad tuvieron víctimas. En este problema es la métrica clave, porque un falso negativo podría retrasar una ambulancia;
- es **fácil de explicar**: sus reglas (*si es atropello → víctimas; si es choque en la madrugada → más riesgo...*) se pueden mostrar a la Secretaría de Tránsito;
- es liviano y rápido para una app web.

La regresión logística tiene mejor Accuracy, pero detecta solo ≈ 35% de los accidentes con víctimas, y el Random Forest queda en un punto intermedio sin superar al árbol en F1. Las métricas son moderadas (Accuracy ≈ 0,65–0,69): con solo hora, día, mes, clase y tipo de vía no se puede saber con certeza si un *choque* deja heridos, porque falta información como el tipo de vehículo o la velocidad.

## 8. Conclusiones
**¿Qué aprendió el modelo?**
- La **clase de accidente** es la variable que más pesa: los **atropellos**, las **caídas de ocupante** (motos) y los **volcamientos** casi siempre dejan víctimas, mientras que la mayoría de choques son solo daños.
- La **hora** importa: en la **madrugada y la noche** el porcentaje de accidentes con víctimas es más alto que en horas de oficina (menos tráfico → más velocidad, más alcohol y menos visibilidad).
- Los **fines de semana**, en especial el domingo, hay menos accidentes, pero una mayor proporción de ellos deja víctimas.
- Los datos mostraron un **cambio en el registro desde 2023** (casi no se reportan accidentes de solo daños). Detectarlo en la exploración evitó entrenar un modelo sesgado.

**¿Qué le dice a la política pública?**
- Reforzar **controles de velocidad y alcoholemia en la noche, la madrugada y los fines de semana**.
- Priorizar **infraestructura para peatones y motociclistas** (pasos seguros, reductores de velocidad, campañas de casco), porque atropellos y caídas de ocupante son los accidentes más graves.
- Con el modelo en una app, una central de emergencias podría **estimar la probabilidad de víctimas** apenas llega el reporte y priorizar ambulancias.

**Limitaciones:**
- El desempeño es **moderado** (F1 ≈ 0,51): el modelo sirve como apoyo para priorizar, no para decidir solo.
- El dataset no trae variables clave como el tipo de vehículo, el clima, la velocidad o la edad de los involucrados; con ellas el modelo mejoraría.
- Las direcciones están escritas sin un estándar, por eso solo se aprovechó el tipo de vía.
- El modelo se entrenó con 2018–2022; para usarlo con datos recientes, la fuente tendría que volver a registrar los accidentes de solo daños.

In [ ]:
# Guardamos el dataset limpio (opcional, útil para la app)
df_limpio = df[cat_cols + num_cols + ["con_victimas"]]
df_limpio.to_csv("accidentalidad_limpio.csv", index=False)
print("Dataset limpio guardado:", df_limpio.shape)